<a href="https://colab.research.google.com/github/DengAchiek/Data-science/blob/main/C10_M05_Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### RNN with Attention (Possible solution)

In [1]:
# -*- coding: utf-8 -*-
"""
Wind Turbine Power Prediction with RNNs + Attention

This lab uses an Attention-enhanced Recurrent Neural Network to predict wind turbine power output.
"""

# Step 0: Load libraries & Data
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, r2_score
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Dense, Dropout, Layer
from tensorflow.keras.callbacks import EarlyStopping
import math

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

# Step 1: Prepare the Data
# Load sample/dataset (e.g., wind turbine power dataset)
# Replace with actual dataset path if loading from CSV: df = pd.read_csv('wind_turbine_data.csv')
date_rng = pd.date_range(start='2023-01-01', periods=1000, freq='H')
df = pd.DataFrame(date_rng, columns=['date'])
df['WindSpeed'] = np.random.normal(loc=8, scale=2, size=len(date_rng))
df['PowerOutput'] = df['WindSpeed'] ** 3 * 0.5 + np.random.normal(0, 10, size=len(date_rng))

# Scale features
scaler = MinMaxScaler(feature_range=(0, 1))
scaled_data = scaler.fit_transform(df[['WindSpeed', 'PowerOutput']].values)

# Step 2: Create sequences
def create_sequences(data, sequence_length):
    X, y = [], []
    for i in range(len(data) - sequence_length):
        X.append(data[i:i + sequence_length])
        y.append(data[i + sequence_length, 1])  # Target variable (PowerOutput)
    return np.array(X), np.array(y)

SEQ_LENGTH = 24
X, y = create_sequences(scaled_data, SEQ_LENGTH)

# Step 3: Split the data
train_size = int(len(X) * 0.8)
X_train, X_test = X[:train_size], X[train_size:]
y_train, y_test = y[:train_size], y[train_size:]

# Step 4: Define a Custom Attention Layer
class AttentionLayer(Layer):
    def __init__(self, **kwargs):
        super(AttentionLayer, self).__init__(**kwargs)

    def build(self, input_shape):
        self.W = self.add_weight(name='attention_weight',
                                 shape=(input_shape[-1], 1),
                                 initializer='glorot_uniform',
                                 trainable=True)
        self.b = self.add_weight(name='attention_bias',
                                 shape=(input_shape[1], 1),
                                 initializer='zeros',
                                 trainable=True)
        super(AttentionLayer, self).build(input_shape)

    def call(self, inputs):
        # inputs shape: (batch_size, time_steps, hidden_dim)
        e = tf.tanh(tf.matmul(inputs, self.W) + self.b)
        a = tf.nn.softmax(e, axis=1)
        output = inputs * a
        return tf.reduce_sum(output, axis=1)

# Step 5: Build the RNN Model with Attention
inputs = Input(shape=(X_train.shape[1], X_train.shape[2]))
lstm_out = LSTM(64, return_sequences=True)(inputs)
lstm_out = Dropout(0.2)(lstm_out)
attention_out = AttentionLayer()(lstm_out)
dense_out = Dense(32, activation='relu')(attention_out)
outputs = Dense(1)(dense_out)

model = Model(inputs=inputs, outputs=outputs)
model.summary()

# Step 6: Compile and Train the Model
model.compile(optimizer='adam', loss='mean_squared_error', metrics=['mae'])

early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

history = model.fit(
    X_train, y_train,
    epochs=50,
    batch_size=32,
    validation_split=0.2,
    callbacks=[early_stopping],
    verbose=1
)

/tmp/ipykernel_1329/3884319857.py:27: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  date_rng = pd.date_range(start='2023-01-01', periods=1000, freq='H')


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 24, 2)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 24, 64)         │        17,152 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 24, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ attention_layer                 │ (None, 64)             │            88 │
│ (AttentionLayer)                │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 19,353 (75.60 KB)

 Trainable params: 19,353 (75.60 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 8s 151ms/step - loss: 0.0166 - mae: 0.0972 - val_loss: 0.0127 - val_mae: 0.0797
Epoch 2/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 50ms/step - loss: 0.0130 - mae: 0.0835 - val_loss: 0.0123 - val_mae: 0.0814
Epoch 3/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 35ms/step - loss: 0.0127 - mae: 0.0834 - val_loss: 0.0123 - val_mae: 0.0805
Epoch 4/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 30ms/step - loss: 0.0128 - mae: 0.0839 - val_loss: 0.0123 - val_mae: 0.0802
Epoch 5/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - loss: 0.0127 - mae: 0.0835 - val_loss: 0.0123 - val_mae: 0.0802
Epoch 6/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - loss: 0.0128 - mae: 0.0838 - val_loss: 0.0123 - val_mae: 0.0801
Epoch 7/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - loss: 0.0128 - mae: 0.0839 - val_loss: 0.0123 - val_mae: 0.0801
Epoch 8/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - loss: 0.0128 - mae: 0.0839 - val_loss: 0.0123 - val_mae: 0.0802
Epoch 9/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - loss: 0.01